# 실시간 문장 모호성 특징 추출기 (Real-time Sentence Ambiguity Feature Extractor)

본 노트북은 VLM 다중 이미지 순서 정렬 파이프라인 아키텍처의 **Phase 1(통사 구조 분류)** 및 **Phase 2(경량 의미론적 특징 추출)** 설계에 맞추어, 임의의 입력 문장에 대한 문법적 특징을 실시간으로 분석하고 추출합니다.

### 사전(Dictionary) 없는 순수 구조적 방어 설계 (과적합 및 과잉 일반화 원천 방지):
- **공백 전처리 (`clean_sentence`)**: 문장 앞뒤 및 중간의 무의미한 공백을 정합하여 파싱 오류를 최소화합니다.
- **순수 구문 구조 기반 동사 복원 (No Hardcoding/No Dictionary)**: 특정 어휘 목록(사전)을 하드코딩하여 과적합을 유발하는 방식 대신, **"ROOT 명사가 Noun Compound/Modifier(compound, nmod) 자식을 가지고 있는가"**라는 순수 문법 구조 규칙을 활용해 동사 품사 오탐을 찾아내고 복원합니다. 진짜 명사구(예: "heavy rain" - 형용사 수식어만 존재)는 안전하게 보존됩니다.
- **중립 가상 주어 (`[unspecified subject]`)**: 문맥상 주어가 생략되어 파서가 잡을 수 없는 엣지 케이스는 바이어스(Bias)가 있는 `someone` 대신 **`[unspecified subject]`** 토큰을 주입합니다. 이는 VLM에게 텍스트 편향을 배제하고 이미지 피처에서 행동 주체를 직접 추론하도록 유도하는 마스킹 토큰 역할을 합니다.

In [ ]:
# 1. 환경 설정 및 SpaCy 모델 로드
try:
    import spacy
except ImportError:
    import subprocess
    import sys
    print("SpaCy 패키지가 없어 설치를 진행합니다...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])
    import spacy
import re

# 영어 경량 의존성 구문 분석 모델 로드
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("en_core_web_sm 모델 다운로드 중...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("SpaCy 환경 및 순수 구조 방어 파서 로드 완료!")

## 2. 노이즈 정제 및 고도화 특징 추출 함수 정의

In [ ]:
def clean_sentence(text):
    """
    문장 맨 앞/뒤의 무의미한 빈칸과 단어 사이의 중복 공백(2칸 이상)을 제거합니다.
    """
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def extract_ambiguity_features(sentence):
    """
    하드코딩 사전 없이 순수 구문 트리 구조를 활용해 엣지 케이스를 안전하게 방어하는 실시간 분석 함수
    """
    cleaned_sentence = clean_sentence(sentence)
    if not cleaned_sentence:
        return {
            "Partition": "Type-1",
            "Pred_Count": 0,
            "Subj_Count": 1,
            "Subject_Words": "[unspecified subject]",
            "Predicate_Words": ""
        }
        
    doc = nlp(cleaned_sentence)
    
    subjs = []
    preds = []
    subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
    
    # 1. 기본 주어 추출 및 등위 관계 확장 (Coordination Expansion)
    for token in doc:
        if token.dep_ in subj_deps:
            subjs.append(token.text)
            for child in token.children:
                if child.dep_ == "conj" and child.pos_ in {"NOUN", "PROPN", "PRON"}:
                    subjs.append(child.text)
                    
    for token in doc:
        if token.pos_ in {"VERB", "AUX"}:
            preds.append(token.text)
            
    # 2. 방어 가드 1: 순수 구문 매핑 기반 동작 서술어 복원 (과적합 유발 어휘 사전 100% 제거)
    if len(preds) == 0:
        root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
        if root_token and root_token.pos_ in {"NOUN", "PROPN"}:
            # ROOT 명사 아래에 명사 compound/nmod가 자식으로 연결되어 있을 경우만 주어-동사 오탐으로 간주하여 승격
            comp = next((t for t in root_token.children if t.dep_ in {"compound", "nmod"}), None)
            if comp:
                preds.append(root_token.text)
                subjs.append(comp.text)
                    
    # 3. 방어 가드 2: 유도부사 구문 보어 주어 승격
    if len(subjs) == 0:
        root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
        if root_token and root_token.pos_ in {"VERB", "AUX"}:
            attr = next((t for t in root_token.children if t.dep_ == "attr"), None)
            if attr:
                subjs.append(attr.text)

    # 4. 명사구 형태의 캡션 (가상 동사 주입 fallback)
    if len(preds) == 0:
        root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
        if root_token:
            preds.append("is shown")
            subjs.append(root_token.text)
            
    # 5. 준동사 주어 상속 (Subject Inheritance)
    if len(subjs) > 0 and subjs[0] != "[unspecified subject]":
        main_subject = subjs[0]
        for token in doc:
            if token.dep_ in {"advcl", "xcomp", "ccomp"} and token.pos_ in {"VERB", "AUX"}:
                has_subj = any(c.dep_ in subj_deps for c in token.children)
                if not has_subj:
                    subjs.append(main_subject)
                    
    # 6. 방어 가드 3: 유·무정성 왜곡 방지를 위한 중립 마스크 주입
    if len(subjs) == 0:
        subjs.append("[unspecified subject]")
        
    # 7. 문장 통사 구조 유형 분류
    has_subordinate_clause = False
    has_parallel_clause = False
    for token in doc:
        if token.dep_ in {"advcl", "ccomp"}:
            has_subordinate_clause = True
        if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
            has_parallel_clause = True
            
    if not has_subordinate_clause and not has_parallel_clause:
        partition = "Type-1"
    elif has_subordinate_clause:
        partition = "Type-2"
    else:
        partition = "Type-3"
        
    return {
        "Partition": partition,
        "Pred_Count": len(preds),
        "Subj_Count": len(subjs),
        "Subject_Words": ", ".join(subjs),
        "Predicate_Words": ", ".join(preds)
    }

## 3. 실시간 테스트 실행

사물 행위 생략 문장과 이중 공백 문장이 정제되어 `[unspecified subject]` 토큰 및 올바른 품사 복원이 유도되는지 검증합니다.

In [ ]:
test_cases = [
    "   A child swings across the monkey bars.   ",  # 품사 오탐 복원 대상
    "Drops to the floor.",  # 사물 유실 주어 -> Animacy 왜곡 방지 대상 (unspecified subject)
    "A sudden heavy rain.",  # 진짜 명사구 -> 오탐 복원 미대상 (is shown 주입)
    "Next there is a little boy shown dipping his hand.",  # 유도부사 보어 승격
    "The boy and girl laugh and jump.",  # 등위 주어 확장
]

print("=== 순수 구조 방어 실시간 테스트 결과 ===")
for s in test_cases:
    res = extract_ambiguity_features(s)
    print(f"\n입력: \"{s}\"")
    print(f" -> Partition: {res['Partition']}")
    print(f" -> 주어 ({res['Subj_Count']}개)  : [{res['Subject_Words']}]")
    print(f" -> 서술어 ({res['Pred_Count']}개): [{res['Predicate_Words']}]")

## 4. 고속 배치 처리 모듈 (`nlp.pipe` 연동)

In [ ]:
def extract_features_batch(sentences, batch_size=256):
    cleaned_sentences = [clean_sentence(s) for s in sentences]
    results = []
    
    for doc in nlp.pipe(cleaned_sentences, batch_size=batch_size):
        if not doc.text.strip():
            results.append({
                "Partition": "Type-1",
                "Pred_Count": 0,
                "Subj_Count": 1,
                "Subject_Words": "[unspecified subject]",
                "Predicate_Words": ""
            })
            continue
            
        # 본문 로직 적용
        subjs = []
        preds = []
        subj_deps = {"nsubj", "nsubjpass", "csubj", "csubjpass"}
        
        for token in doc:
            if token.dep_ in subj_deps:
                subjs.append(token.text)
                for child in token.children:
                    if child.dep_ == "conj" and child.pos_ in {"NOUN", "PROPN", "PRON"}:
                        subjs.append(child.text)
                        
        for token in doc:
            if token.pos_ in {"VERB", "AUX"}:
                preds.append(token.text)
                
        if len(preds) == 0:
            root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
            if root_token and root_token.pos_ in {"NOUN", "PROPN"}:
                comp = next((t for t in root_token.children if t.dep_ in {"compound", "nmod"}), None)
                if comp:
                    preds.append(root_token.text)
                    subjs.append(comp.text)
                        
        if len(subjs) == 0:
            root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
            if root_token and root_token.pos_ in {"VERB", "AUX"}:
                attr = next((t for t in root_token.children if t.dep_ == "attr"), None)
                if attr:
                    subjs.append(attr.text)

        if len(preds) == 0:
            root_token = next((t for t in doc if t.dep_ == "ROOT"), None)
            if root_token:
                preds.append("is shown")
                subjs.append(root_token.text)
                
        if len(subjs) > 0 and subjs[0] != "[unspecified subject]":
            main_subject = subjs[0]
            for token in doc:
                if token.dep_ in {"advcl", "xcomp", "ccomp"} and token.pos_ in {"VERB", "AUX"}:
                    has_subj = any(c.dep_ in subj_deps for c in token.children)
                    if not has_subj:
                        subjs.append(main_subject)
                        
        if len(subjs) == 0:
            subjs.append("[unspecified subject]")
            
        has_subordinate_clause = False
        has_parallel_clause = False
        for token in doc:
            if token.dep_ in {"advcl", "ccomp"}:
                has_subordinate_clause = True
            if token.dep_ == "conj" and token.pos_ in {"VERB", "AUX"}:
                has_parallel_clause = True
                
        if not has_subordinate_clause and not has_parallel_clause:
            partition = "Type-1"
        elif has_subordinate_clause:
            partition = "Type-2"
        else:
            partition = "Type-3"
            
        results.append({
            "Partition": partition,
            "Pred_Count": len(preds),
            "Subj_Count": len(subjs),
            "Subject_Words": ", ".join(subjs),
            "Predicate_Words": ", ".join(preds)
        })
    return results